## <u>Observability</u>

**Observability** is the ability to **understand what is happening inside a system by analyzing its outputs such as logs, metrics, and traces**.

It helps developers **detect issues, debug problems, and monitor performance without directly inspecting the internal code or state of the system**.


### Why Observability for LLMs

If an LLM gives a wrong answer in production, traditional logs are not enough.
We need observability for:
1. Debugging unpredictable outputs
2. Monitoring performance and latency
3. Tracking costs and token usage
4. Detecting prompt or model degradation

Eg: 

RAG Pipeline Flow
```
+--------------+     +---------------+     +---------------------+     +------------+     +-----------+
| User Request | --> | RAG Retrieval | --> | Prompt Construction | --> |  LLM Call  | --> | Response  |
+--------------+     +---------------+     +---------------------+     +------------+     +-----------+
```

- Self Hosted vs Vendor locked

### Tools

1- LangSmith <br>
2- Langfuse (O/S)<br>
2- Netra <br>
3- Helicone <br>

## Langfuse
Fully open-source (MIT license)

Self-hostable (good for security & privacy)

### What Langfuse captures:
1. Inputs
2. Outputs
3. Metadata
4. Latency
5. Cost & Tokens usage
6. Traces

### Local setup

1. Host langfuse locally

In [ ]:
%%bash
git clone https://github.com/langfuse/langfuse.git
cd langfuse
docker compose up -d

2. Open [Langfuse Dashboard](http://localhost:3000), sign up and log in. Then create a New Organization.
Under Projects, create a new project, and in Tracing, generate a new API key.

Finally, store the following values in your .env file:
```
LANGFUSE_SECRET_KEY=XXX
LANGFUSE_PUBLIC_KEY=XXX
LANGFUSE_BASE_URL=http://localhost:3000
```

In [ ]:
from dotenv import load_dotenv
from langfuse.openai import OpenAI
import os

load_dotenv()

LITELLM_API_KEY = os.getenv("LITELLM_API_KEY")
LITELLM_BASE_URL = os.getenv("LITELLM_BASE_URL")
client = OpenAI(
    api_key=LITELLM_API_KEY,
    base_url=LITELLM_BASE_URL,
)

system_prompt = "You are a history assistant. Answer using verified historical facts only."
query = "What happened in poland during 1989"

def run_llm(query):
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    )

    return response.choices[0].message.content


print(run_llm(query))

#### Metrics to be noted

1. Latency
2. Env
3. Cost
4. Token usage
5. Model
6. Temperature
7. max_tokens
8. top_p
9. frequency_penalty
10. presence_penalty
11. S/m prompt
12. User prompt
13. Response
14. Metadata

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langfuse.langchain import CallbackHandler
from dotenv import load_dotenv

load_dotenv()

# 1. Setup Langfuse Callback
langfuse_handler = CallbackHandler()

# 2. Define Graph State
class State(TypedDict):
    input_text: str
    sentiment: str
    final_response: str

# 3. Define Nodes
def analyze_sentiment(state: State):
    # Logic to "analyze" (mocking LLM call)
    text = state["input_text"].lower()
    sentiment = "positive" if "good" in text else "neutral/negative"
    return {"sentiment": sentiment}

def format_response(state: State):
    response = f"I've analyzed your text. It feels {state['sentiment']}!"
    return {"final_response": response}

# 4. Build the Graph
builder = StateGraph(State)
builder.add_node("analyzer", analyze_sentiment)
builder.add_node("formatter", format_response)

builder.add_edge(START, "analyzer")
builder.add_edge("analyzer", "formatter")
builder.add_edge("formatter", END)

graph = builder.compile()

# 5. Run with Langfuse Tracking
inputs = {"input_text": "Today is a bad day!"}
config = {"callbacks": [langfuse_handler]}

result = graph.invoke(inputs, config=config)
print(result["final_response"])

### Logging inputs/outputs safely (PII considerations)

In [ ]:
from langfuse import observe

system_prompt = "You are a strong password generator tool"
query = "My email is: joan@gmail.com. can you generate a strong password for this email ?"

@observe()
def run_llm(query):
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ]
    )

    return response.choices[0].message.content


print(run_llm(query))

#### PII leakage scenarios in LLM observability

1. Users input sensitive data into prompts
2. RAG systems retrieving sensitive documents
3. LLM outputs containing sensitive information

#### How to avoid PII Leakage

##### 1. Redact the input query before sending it to the LLM.

In [ ]:
import re

def redact_pii(text: str):
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)
    text = re.sub(r'\b\d{10}\b', '[PHONE]', text)
    return text

query = redact_pii(query)
print(run_llm(query))

##### 2. Redact only the Langfuse logs using the `mask` property

Perform an anonymous Langfuse instantiation to register the masking method before using any @observe() decorators.

In [ ]:
from langfuse import Langfuse, observe

def redact_pii(data, **kwargs):
    """Checks data for sensitive PII and redacts any identified content"""
    pass
 
Langfuse(mask=redact_pii)

@observe()
def run_llm(query):
    pass

Masking method using a 3rd party NLP

Packages required: presidio-analyzer presidio-anonymizer spacy

In [ ]:
%%bash
python3 -m spacy download en_core_web_lg